# Limit order book & imbalance modeling with `quantstats.lob`

This notebook records a few minutes of live 10-level order book snapshots from
Binance's public market-data API (no key needed) and runs every model in
`quantstats.lob` on them:

1. book features and the order book ladder,
2. **queue imbalance** as a predictor of the next price move,
3. **order flow imbalance (OFI)** and linear price impact,
4. Stoikov's **micro-price**, fitted on the first half and tested on the second.

If Binance can't be reached from where you run this, it falls back to a
simulated book so every cell still runs.

Large-tick instruments (spread usually one tick, deep queues) are where
imbalance models shine, so the default symbol is `DOGEUSDT`. Try `BTCUSDT` to
see a small-tick market where the micro-price adds nothing.

In [ ]:
import time

import numpy as np
import pandas as pd
import requests

import quantstats as qs
from quantstats import lob

SYMBOL = "DOGEUSDT"
SECONDS = 300      # how long to record
LEVELS = 10

## 1. Record order book snapshots

`lob.from_depth` turns exchange depth snapshots (`{"bids": [[price, size], ...], "asks": [...]}`)
into the book layout every `lob` function takes: one row per snapshot with
`bid_price_1`, `bid_size_1`, `ask_price_1`, `ask_size_1`, ... columns.

In [ ]:
URL = "https://data-api.binance.vision/api/v3/depth"


def record(symbol, seconds, levels=10, interval=0.2):
    """Poll REST depth snapshots, keeping each new book once."""
    session, snaps, stamps, last = requests.Session(), [], [], None
    end = time.time() + seconds
    while time.time() < end:
        t0 = time.time()
        d = session.get(URL, params={"symbol": symbol, "limit": levels}, timeout=5).json()
        if d["lastUpdateId"] != last:
            last = d["lastUpdateId"]
            snaps.append(d)
            stamps.append(pd.Timestamp(t0, unit="s"))
        time.sleep(max(0.0, interval - (time.time() - t0)))
    return lob.from_depth(snaps, timestamps=stamps)


def simulate(n=6000, tick=0.01, seed=7):
    """Fallback: a symmetric book where the thinner queue tends to empty first."""
    rng = np.random.default_rng(seed)
    bid, ask, qb, qa, rows = 10000, 10001, 50.0, 50.0, []
    for _ in range(n):
        qb += rng.normal(0, 8); qa += rng.normal(0, 8); wide = ask - bid > 1
        if qa <= 0:
            ask += 1; bid += wide; qa = rng.uniform(20, 80); qb = rng.uniform(5, 30) if wide else qb
        elif qb <= 0:
            bid -= 1; ask -= wide; qb = rng.uniform(20, 80); qa = rng.uniform(5, 30) if wide else qa
        elif wide and rng.random() < 0.5:
            if rng.random() < 0.5: bid += 1; qb = rng.uniform(5, 30)
            else: ask -= 1; qa = rng.uniform(5, 30)
        rows.append({"bids": [[(bid - i) * tick, qb if i == 0 else 70.0] for i in range(3)],
                     "asks": [[(ask + i) * tick, qa if i == 0 else 70.0] for i in range(3)]})
    return lob.from_depth(rows, timestamps=pd.date_range("2024-01-02", periods=n, freq="100ms"))


try:
    book = record(SYMBOL, SECONDS, LEVELS)
    source = f"Binance {SYMBOL}"
except Exception as exc:  # network blocked, geo-restricted, ...
    print(f"Live data unavailable ({exc!r}); using a simulated book.")
    book, source = simulate(), "simulated book"

print(f"{source}: {len(book):,} snapshots over {book.index[-1] - book.index[0]}, {lob.levels(book)} levels")
book.iloc[:3, :8]

## 2. Book features

In [ ]:
tick = lob._infer_tick(lob.spread(book))
features = pd.DataFrame({
    "mid": lob.mid_price(book),
    "spread_ticks": (lob.spread(book) / tick).round(),
    "weighted_mid": lob.weighted_mid(book),
    "queue_imb": lob.queue_imbalance(book),
    "depth_imb_5": lob.depth_imbalance(book, levels=min(5, lob.levels(book)), decay=0.5),
    "ofi": lob.order_flow_imbalance(book),
})
print(f"tick = {tick:g}; spread is one tick {100 * (features.spread_ticks == 1).mean():.1f}% of the time")
features.describe().T[["mean", "std", "min", "max"]]

In [ ]:
qs.plots.lob_depth(book)

In [ ]:
qs.plots.lob_timeseries(book.iloc[-300:])

## 3. Does imbalance predict the next move?

For each imbalance bucket, how often is the **next** mid-price change up or down?
A useful signal shows P(up) rising steadily from left to right.

In [ ]:
signal = lob.imbalance_signal(book, bins=5)
signal.round(3)

In [ ]:
qs.plots.lob_imbalance_signal(signal)

## 4. Order flow imbalance and price impact

OFI nets out bid-side additions against ask-side additions (and the reverse for
cancellations and trades). Cont, Kukanov & Stoikov (2014) show mid changes over
short windows are close to linear in OFI.

In [ ]:
impact = lob.price_impact(book, freq="10s")
impact_deep = lob.price_impact(book, freq="10s", levels=min(5, lob.levels(book)), normalize_depth=True)
iofi = lob.integrated_ofi(book, freq="10s", levels=min(5, lob.levels(book)))
print(f"best level: beta={impact.beta:.3g}, t={impact.beta_tstat:.1f}, R2={impact.r2:.2f}, n={impact.nobs}")
print(f"5 levels, depth-normalised: R2={impact_deep.r2:.2f}")
print("integrated OFI level weights:", iofi.weights.round(3).to_dict())

In [ ]:
qs.plots.lob_price_impact(impact)

## 5. Micro-price, tested out of sample

Fit Stoikov's micro-price on the first half of the recording, then check on the
second half which price best predicts the **next different mid**: the mid, the
size-weighted mid, or the micro-price.

In [ ]:
half = len(book) // 2
train, test = book.iloc[:half], book.iloc[half:]
model = lob.fit_microprice(train, tick_size=tick, n_imbalance=6, n_spread=2)
(model.adjustment / tick).round(3)

In [ ]:
mid = lob.mid_price(test)
changed = mid.diff().fillna(1) != 0
run = changed.cumsum().to_numpy() - 1
target = pd.Series(mid[changed].shift(-1).to_numpy()[run], index=mid.index)
ok = target.notna()

def mse(pred):
    return float((((target - pred)[ok]) / tick).pow(2).mean())

scores = pd.Series({
    "mid": mse(mid),
    "weighted mid": mse(lob.weighted_mid(test)),
    "micro-price": mse(model.predict(test)),
}, name="out-of-sample MSE (ticks^2)")
print(f"micro-price cuts the error of the mid by {100 * (1 - scores['micro-price'] / scores['mid']):.1f}%")
scores.round(4).to_frame()

In [ ]:
qs.plots.lob_microprice(model)

## Reading the results

* **Imbalance** is the most reliable short-term signal: on large-tick books the
  probability of an up-move typically climbs from well under 0.5 to well over 0.5
  across the buckets.
* **OFI** usually explains a large share of short-window price changes; adding
  deeper levels and normalising by depth tends to help.
* The **micro-price** beats the mid on large-tick books, where the spread is
  mostly one tick and the mid moves in half-tick steps. On small-tick books
  (BTCUSDT) the mid jumps many ticks between snapshots and the adjustment,
  a fraction of a tick, does not matter.

References: Cont, Kukanov & Stoikov (2014), *The price impact of order book
events*; Cont, Cucuringu & Zhang (2023), *Cross-impact of order flow imbalance*;
Gould & Bonart (2016), *Queue imbalance as a one-tick-ahead price predictor*;
Stoikov (2018), *The micro-price*.